# Text chapterize

In this example, we use an LLM because of its flexibility and ease of use. It allows us to complete the task without training a model. But note that for very structured outputs, a simple classification model could also be trained once enough samples are collected.

We can take the following approach:

1. Create a clever prompt
2. Give the unstructured text to an LLM
3. Retrieve the structured output from the LLM
4. (Save it to a database)

<img title="NER" alt="example of NER" src="https://xebia.com/wp-content/uploads/2023/09/archetype-llm-batch-use-case-basic-prompting-1.jpg.webp">

Text chapterization is the task of dividing a long text into smaller segments or chapters, based on the structure, topic, and coherence of the text. Text chapterization can help readers navigate and comprehend the text more easily, and can also facilitate text summarization, indexing, and retrieval.  

Large Language Models (LLMs) are neural network models that can generate natural language texts based on a given input or context. LLMs can also be used for text chapterization, by leveraging their ability to capture the semantic and syntactic relationships between sentences and paragraphs.  

There are different methods and challenges for using LLMs for text chapterization, depending on the type, length, and domain of the text, as well as the desired number, size, and quality of the chapters.   

How to Extract Structured Data from Unstructured Text using LLMs: Here we will explain how to use LLMs to transform unstructured text into structured data, such as extracting topics.  
How to Label Text Data Using LLMs?  
LLMParser: This is a web tool that allows you to classify and extract text with LLMs, such as identifying entities, relations, and intents from natural language texts.
A Step-By-Step Guide to Evaluating an LLM Text Summarization Task.  
How to use LangChain to Classify Text Using an LLM

## installing needed libraries

In [ ]:
!pip install  tiktoken langchain pandas unstructured
! pip install transformers huggingface_hub sentence-transformers

## importing libraries

In [ ]:
from langchain.chains import (
    StuffDocumentsChain,
    LLMChain,
    ConversationalRetrievalChain,
)
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from langchain_community.llms import Ollama, HuggingFaceHub

import os
from google.colab import userdata

os.environ["HUGGINGFACEHUB_API_TOKEN"] = userdata.get('HF_token')
os.environ["HF_TOKEN"] = userdata.get('HF_token')

## split data
convert the user docs into shape that LLM can understand 'document type'

In [ ]:
def split_docs(text):
  splited_text = RecursiveCharacterTextSplitter(separators = [".", "\n", " "], chunk_size=3000, chunk_overlap=0)
  docs = splited_text.create_documents([text])
  return docs

## process data
  * First we create prompt template to tell the LLM model what to do
  * Then, specify the task we need from the LLM model
  * Finally, call the LLM model to perform the task

In [ ]:
llm = HuggingFaceHub(
  repo_id="HuggingFaceH4/zephyr-7b-beta",
  task="text-generation",
)

# llm = HuggingFaceHub(
#   repo_id="microsoft/phi-2",
#   task="text-generation",
# )


In [ ]:

template = """أعط عنوانًا واحدا قصيرًا جدًا. أقل من 10 كلمات فقط لهذه الفقرة:
"{text}"
أجب بعنوان واحد فقط
العنوان: """
SUMMARIZE_PROMPT = PromptTemplate.from_template(template)# Run chain

llm_chain = LLMChain(llm=llm, prompt=SUMMARIZE_PROMPT)
stuff_chain = StuffDocumentsChain(llm_chain=llm_chain, document_variable_name="text")


In [ ]:
docs = "".join(open("/content/drive/MyDrive/NLP/NLU/NLU/data/semantic_search/en.txt").readlines())
results = stuff_chain.run(split_docs(docs))


In [ ]:
results.split('\n')